# 041: Python Tuples — Complete Reference, Immutability Internals, Hashing, and Memory Optimization

## 1. WHAT IS A PYTHON TUPLE?

A Python `tuple` is an **ordered, immutable, heterogeneous** sequence type. Once created, its elements cannot be changed, added, or removed.

### CPython Internal Representation (`tupleobject.h`)
```c
typedef struct {
    PyObject_VAR_HEAD          // ob_refcnt, ob_type, ob_size
    PyObject *ob_item[1];      // Inline array of PyObject* pointers
} PyTupleObject;
```
- Unlike lists, the pointer array is stored **inline** within the tuple object (not via an external pointer). This means tuples have no `allocated` field and no over-allocation — they use exactly as much memory as needed.
- **Memory advantage**: A tuple of $n$ elements is 56 + 8n bytes vs a list's 56 + 8n + overhead for `allocated` tracking + over-allocated spare slots.

### Tuple Memory Comparison with Lists
- Empty tuple: **40 bytes** (vs 56 bytes for empty list).
- Tuple(1 elem): **48 bytes** (vs 64+ bytes for list with 4 preallocated slots).
- **CPython caches** small tuples: The empty tuple `()` is a singleton. Tuples of length 1–20 are cached via a **free list** to avoid repeated memory allocation.

### Immutability Guarantee
- The tuple's `ob_item` array is fixed at creation time. There is no `__setitem__`, `__delitem__`, `append`, or `insert` method.
- **Caveat**: If a tuple contains a mutable object (like a list), the mutable object itself CAN be modified. The tuple's reference (pointer) to that object remains unchanged, but the object's internal state can change.

---


In [ ]:
import sys

print("=" * 70)
print("SECTION 1: Tuple vs List Memory Comparison")
print("=" * 70)

# Compare memory footprints
for n in [0, 1, 3, 5, 10, 100]:
    t = tuple(range(n))
    l = list(range(n))
    print(f"n={n:>3} | tuple: {sys.getsizeof(t):>5} bytes | list: {sys.getsizeof(l):>5} bytes | "
          f"savings: {sys.getsizeof(l) - sys.getsizeof(t):>3} bytes")



---
## 2. COMPLETE METHOD REFERENCE

Tuples have only **two** built-in methods: `count()` and `index()`.
This is by design — immutability means no mutation methods exist.

---
### 2.1 `tuple.count(x)`
- **What it does**: Returns the number of times value `x` appears.
- **Returns**: `int`.
- **Time**: $O(n)$ — must scan all elements.
- **Uses `==` for comparison** (not `is`).


In [ ]:
print("\n" + "=" * 70)
print("METHOD: tuple.count(x)")
print("=" * 70)

t = (1, 2, 3, 2, 2, 4, 1)
print(f"(1,2,3,2,2,4,1).count(2):   {t.count(2)}")
print(f"count(5) — not present:     {t.count(5)}")
print(f"count(None) in (None,1,None): {(None, 1, None).count(None)}")

# Edge case: counting in nested tuples — does NOT recurse
nested = ((1, 2), (3, 4), (1, 2))
print(f"count((1,2)) in nested:     {nested.count((1, 2))}")  # Counts tuple equality

# count uses __eq__
print(f"(1.0, 2).count(1):          {(1.0, 2).count(1)}")  # 1.0 == 1 is True!



### 2.2 `tuple.index(x[, start[, end]])`
- **What it does**: Returns the index of the first occurrence of `x` within `tuple[start:end]`.
- **Returns**: `int`.
- **Raises**: `ValueError` if not found.
- **Time**: $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: tuple.index(x, start, end)")
print("=" * 70)

colors = ("red", "green", "blue", "green", "yellow")
print(f"index('green'):              {colors.index('green')}")
print(f"index('green', 2):           {colors.index('green', 2)}")  # Start from index 2

try:
    colors.index("purple")
except ValueError as e:
    print(f"index('purple'):             ValueError: {e}")



---
## 3. TUPLE PACKING AND UNPACKING

- **Packing**: Creating a tuple by grouping values: `t = 1, 2, 3` (parentheses optional).
- **Unpacking**: Assigning tuple elements to variables: `a, b, c = t`.
- **Extended unpacking** (Python 3): `a, *rest = (1, 2, 3, 4)` captures excess elements in a list.
- **Swap idiom**: `a, b = b, a` uses tuple packing/unpacking under the hood.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Tuple Packing & Unpacking")
print("=" * 70)

# Packing (parentheses optional)
packed = 1, 2, 3
print(f"Packed without parens:       {packed}, type: {type(packed)}")

# Single-element tuple REQUIRES trailing comma
single = (42,)
not_a_tuple = (42)
print(f"(42,) is tuple:              {type(single)}")
print(f"(42) is NOT tuple:           {type(not_a_tuple)}")

# Unpacking
a, b, c = (10, 20, 30)
print(f"Unpacked a={a}, b={b}, c={c}")

# Extended unpacking with *
first, *middle, last = (1, 2, 3, 4, 5)
print(f"first={first}, middle={middle}, last={last}")

# Swap idiom
x, y = 5, 10
x, y = y, x
print(f"After swap: x={x}, y={y}")

# Edge case: unpacking with wrong number of variables
try:
    a, b = (1, 2, 3)
except ValueError as e:
    print(f"Wrong unpack count:          ValueError: {e}")

# Nested unpacking
(a, b), (c, d) = (1, 2), (3, 4)
print(f"Nested unpack: a={a}, b={b}, c={c}, d={d}")

# Unpacking in function returns
def minmax(lst):
    return min(lst), max(lst)

lo, hi = minmax([3, 1, 4, 1, 5])
print(f"minmax returns tuple:        lo={lo}, hi={hi}")



---
## 4. IMMUTABILITY DEEP DIVE

### What "immutable" actually means
- The tuple's **pointer array** is fixed. You cannot reassign `t[0] = x`.
- But if `t[0]` points to a mutable object, that object CAN be mutated.
- The tuple's **hash** depends on all elements being hashable. If it contains an unhashable element (like a list), the tuple itself becomes unhashable.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Immutability Deep Dive")
print("=" * 70)

# Cannot assign to tuple elements
t = (1, 2, 3)
try:
    t[0] = 99
except TypeError as e:
    print(f"t[0] = 99:                   TypeError: {e}")

# Mutable object inside a tuple CAN be mutated
t = ([1, 2], [3, 4])
t[0].append(99)  # This works!
print(f"Mutated inner list:          {t}")

# += on a tuple element (the augmented assignment trap)
t = ([1, 2],)
try:
    t[0] += [3, 4]
except TypeError as e:
    print(f"t[0] += [3,4] raises:        TypeError: {e}")
    print(f"But list WAS mutated:        {t}")
    # The list's __iadd__ succeeded, but tuple's __setitem__ failed
    # This is one of Python's most famous gotchas!



---
## 5. HASHING AND TUPLES AS DICTIONARY KEYS

- A tuple is **hashable** if and only if ALL its elements are hashable.
- Hashable tuples can be used as dictionary keys and set elements.
- CPython computes the tuple hash using a modified **FNV-1a** hash that XORs and multiplies each element's hash.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Hashing and Tuples as Dict Keys")
print("=" * 70)

# Hashable tuple
t = (1, 2, 3)
print(f"hash((1,2,3)):               {hash(t)}")

# Tuple as dict key
coords = {(0, 0): "origin", (1, 0): "right", (0, 1): "up"}
print(f"coords[(1,0)]:               {coords[(1, 0)]}")

# Tuple as set element
points = {(0, 0), (1, 1), (0, 0)}  # Duplicate removed
print(f"Set of tuples:               {points}")

# Unhashable tuple (contains a list)
try:
    hash(([1, 2], 3))
except TypeError as e:
    print(f"hash(([1,2], 3)):            TypeError: {e}")

# Tuple hash depends on element ORDER
print(f"hash((1,2)) == hash((2,1)):  {hash((1, 2)) == hash((2, 1))}")



---
## 6. TUPLE CONCATENATION AND REPETITION INTERNALS

- **Concatenation** `t1 + t2`: Creates a **new** tuple. $O(n + m)$.
- **Repetition** `t * k`: Creates a new tuple with elements repeated $k$ times. $O(nk)$.
- Since tuples are immutable, every "modification" creates a new object.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Concatenation & Repetition")
print("=" * 70)

t1 = (1, 2, 3)
t2 = (4, 5, 6)
t3 = t1 + t2
print(f"(1,2,3) + (4,5,6):          {t3}")
print(f"t1 is still:                 {t1}")  # Unchanged

# Repetition
t4 = (0,) * 5
print(f"(0,) * 5:                    {t4}")

# Identity check: concatenation always creates new tuple
print(f"t1 + () is t1:               {(t1 + ()) is t1}")  # Usually False
# But CPython may optimize empty-tuple concat in some versions

# Memory cost of repeated concatenation
import time
t0 = time.perf_counter()
result = ()
for i in range(10000):
    result = result + (i,)  # Creates a new tuple EVERY iteration!
t1_time = time.perf_counter()
print(f"10K concatenations:          {(t1_time - t0) * 1000:.1f} ms (O(n^2) total!)")



---
## 7. NAMED TUPLES — EXTENDING TUPLES WITH FIELD NAMES

`collections.namedtuple` creates tuple subclasses with named fields:
- Provides readability of classes with the efficiency of tuples.
- Still immutable.
- Still hashable.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Named Tuples")
print("=" * 70)

from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(f"Point(3, 4):                 {p}")
print(f"p.x = {p.x}, p.y = {p.y}")
print(f"p[0] = {p[0]}")  # Still indexable
print(f"isinstance(p, tuple):        {isinstance(p, tuple)}")

# _asdict, _replace, _fields
print(f"_asdict():                   {p._asdict()}")
print(f"_replace(x=10):              {p._replace(x=10)}")
print(f"_fields:                     {Point._fields}")



---
## 8. PERFORMANCE COMPARISON: TUPLE vs LIST

Tuples are faster than lists for:
1. **Creation**: Tuple literals are stored as constants in bytecode. Lists must be built at runtime.
2. **Iteration**: Slightly faster due to fixed size (no over-allocation bookkeeping).
3. **Memory**: Smaller footprint (no `allocated` field, no spare slots).


In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: Tuple vs List Performance Benchmark")
print("=" * 70)

import time

N = 1_000_000

# Creation speed
t0 = time.perf_counter()
for _ in range(N):
    x = (1, 2, 3, 4, 5)
t1 = time.perf_counter()
tuple_create = (t1 - t0) * 1000

t0 = time.perf_counter()
for _ in range(N):
    x = [1, 2, 3, 4, 5]
t1 = time.perf_counter()
list_create = (t1 - t0) * 1000

print(f"Tuple creation (1M):         {tuple_create:.1f} ms")
print(f"List creation (1M):          {list_create:.1f} ms")
print(f"Tuple is {list_create / (tuple_create + 0.001):.1f}x faster to create")

# Iteration speed
big_t = tuple(range(100000))
big_l = list(range(100000))

t0 = time.perf_counter()
for _ in big_t:
    pass
t1 = time.perf_counter()
tuple_iter = (t1 - t0) * 1000

t0 = time.perf_counter()
for _ in big_l:
    pass
t1 = time.perf_counter()
list_iter = (t1 - t0) * 1000

print(f"Tuple iteration (100K):      {tuple_iter:.2f} ms")
print(f"List iteration (100K):       {list_iter:.2f} ms")



---
## 9. FAANG & RESEARCH INTERVIEW QUESTIONS

### Q1: Why are tuples faster than lists for creation?
**A**: Tuple literals like `(1, 2, 3)` are compiled as constants in CPython bytecode (the `LOAD_CONST` instruction loads a pre-built tuple object). List literals `[1, 2, 3]` must be constructed at runtime using `BUILD_LIST` + individual `LOAD_CONST` for each element.

### Q2: Can you use a tuple containing a list as a dictionary key?
**A**: No. A tuple is hashable only if ALL its elements are hashable. Lists are unhashable (mutable), so `hash(([1,2], 3))` raises `TypeError`.

### Q3: Explain the `tuple.__iadd__` gotcha with mutable elements.
**A**: When you write `t[0] += [3, 4]` where `t[0]` is a list, Python first calls `list.__iadd__` (which succeeds, mutating the list in-place), then attempts `tuple.__setitem__` (which fails with TypeError). The net result: the list IS mutated, but the exception IS raised. This is a famous Python gotcha.

### Q4: What is tuple interning in CPython?
**A**: CPython maintains a free list of recently deallocated tuple objects (up to length 20). When a new tuple of the same size is needed, CPython reuses the memory from the free list instead of calling `malloc`. The empty tuple `()` is a permanent singleton.

### Q5: When should you prefer a tuple over a list?
**A**: Use tuples when: (1) the data is fixed and should not change (immutability as documentation), (2) you need the data as a dict key or set element (hashability), (3) you want slightly better performance and memory usage, (4) you're returning multiple values from a function.
